# 2. Atactic polystyrene: stereocenters survive initialization

Soft DPD repulsion lets atoms pass through each other, so a tetrahedral
center can invert. FlowerMD's stereochemistry guard records the handedness of
every center and holds it with a native dihedral restraint; the run record
audits the centers before DPD, after DPD and after FIRE. Here we also check
them after the unbiased Sage minimization.

Whole chains are placed as rigid units (`unit="chain"`) because a
polystyrene backbone center has a neighbour in the next repeat.

The exported residues carry the tacticity: `PSR` for monomers built from the
R enantiomer, `PSS` for the S enantiomer, so coloring by residue name shows
the atactic sequence.

In [1]:
import hoomd

# GPU if one is visible, otherwise CPU. OpenMM follows the same choice.
try:
    DEVICE = hoomd.device.GPU()
    OPENMM_PLATFORM = "CUDA"
except Exception:
    DEVICE = hoomd.device.CPU()
    OPENMM_PLATFORM = "CPU"
print(DEVICE, OPENMM_PLATFORM)

<hoomd.device.CPU object at 0x713da83d0320> CPU


In [2]:
# The frozen all-atom PhantomWalk protocol, written out explicitly.
PROTOCOL = dict(
    bonded="uff", A=1250.0, gamma=200.0, kT=1.0, r_cut=3.5, bonded_scale=30.0,
    epsilon_weighting=True, protect_stereochemistry=True, stereo_k=30000.0,
)
RUN = dict(
    dpd_min_steps=3500, dpd_chunk=500, dpd_max_steps=40000, energy_tol=0.02,
    consecutive=2, dpd_samples_per_chunk=5, fire_steps=100, fire_dt=0.001,
    require_convergence=False,
)
DT = 0.001

In [3]:
from pathlib import Path

# Everything this notebook writes goes here.
OUT = Path("outputs/2-atactic-polystyrene")
OUT.mkdir(parents=True, exist_ok=True)

# True saves a DPD + FIRE trajectory (one GSD frame every 250 steps) for a
# movie; False writes only the starting frame.
SAVE_TRAJECTORY = True


def writers(folder):
    """Simulation keywords that put the GSD trajectory and log in `folder`."""
    folder.mkdir(parents=True, exist_ok=True)
    return dict(
        gsd_file_name=str(folder / "dpd.gsd"),
        gsd_write_freq=250 if SAVE_TRAJECTORY else 10**9,
        log_file_name=str(folder / "log.txt"),
    )

In [4]:
import unyt as u
from flowermd.library import AllAtomDPD, AllAtomLattice, AllAtomPhantomWalk, PolyStyrene
from flowermd.internal.stereochemistry import audit_stereochemistry
from phantomwalk.all_atom import sage_handoff

chains = PolyStyrene(lengths=20, num_mols=8, tacticity="atactic", seed=3)
system = AllAtomLattice(chains, density=1.04 * u.g / u.cm**3, unit="chain", seed=3)
ff = AllAtomDPD(system.system, **PROTOCOL)
print(f"{ff.stereo_centers} stereocenters recorded")
sim = AllAtomPhantomWalk.from_system(system, forcefield=ff, dt=DT, device=DEVICE, seed=3,
                                     **writers(OUT))
record = sim.run_initialization(**RUN)
for stage, audit in record["stereochemistry"].items():
    print(f"{stage:>9}: {audit['n_centers']} centers, inverted {audit['inverted_count']}, "
          f"passed {audit['passed']}")

2026-09-24 13:55:35,271 - mbuild.compound - WARNING - Compound.box.lengths < Compound.boundingbox.lengths. There may be particles outside of the defined simulation box.


*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


...

*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


*Warning*: dihedral.harmonic: specified K <= 0


  initial: 152 centers, inverted 0, passed True
 post_dpd: 152 centers, inverted 0, passed True
post_fire: 152 centers, inverted 0, passed True


In [5]:
import numpy as np

box_a = np.asarray(ff.frame.configuration.box[:3])
handoff, minimized_a = sage_handoff(sim.to_compound(), box_a / 10, platform=OPENMM_PLATFORM)
after = audit_stereochemistry(ff.stereo_reference, minimized_a, box_lengths=box_a,
                              planar_tolerance=ff.stereo_planar_tolerance)
print(f"after Sage minimization: inverted {after['inverted_count']}, passed {after['passed']}; "
      f"energy removed {handoff['energy_removed_sage_epsilon_atom']:.2f} eps_max per atom")
sim.write_record(OUT / "record.json")

after Sage minimization: inverted 0, passed True; energy removed 3.69 eps_max per atom


## Export the structures

`residue_topology` reads the FlowerMD hierarchy (melt, molecule, monomer)
into PDB labels: one residue per monomer, one segment ID per molecule (four
base-36 characters, up to 1.68 million molecules), CONECT records for every
bond and the box in CRYST1. Molecules are written whole with their centroids
in the box, so per-chain quantities such as Rg work directly. Three stages
are written:

| file | coordinates |
|---|---|
| `placement.pdb` | the lattice placement before DPD |
| `initialized.pdb` | after DPD + FIRE, what `sim.to_compound()` hands off |
| `minimized.pdb` | after the Sage 2.3.0 minimization; **use this one for analysis** |

`dpd.dcd` is the DPD + FIRE trajectory (whole, continuous molecules) with
`initialized.pdb` as its topology; `movie.pml` and `minimized.pml` open them
in PyMOL (`cd` into the output folder, then `pymol movie.pml`).

The PyMOL scripts color carbons by residue name here, so R and S monomers
differ.

In [6]:
from phantomwalk.all_atom import (
    flush_trajectory, frame_positions, residue_topology, write_pdb, write_trajectory,
)
from phantomwalk.all_atom.visualization import write_pymol_script


def export(folder, sim, ff, minimized_a):
    """Write the three stages, the DPD trajectory and the PyMOL scripts."""
    top = residue_topology(sim.system.system, box_a=ff.frame.configuration.box[:3])
    initialized_a = sim.final_positions() * 10
    write_pdb(top, frame_positions(ff.frame), folder / "placement.pdb")
    write_pdb(top, initialized_a, folder / "initialized.pdb")
    write_pdb(top, minimized_a, folder / "minimized.pdb")
    flush_trajectory(sim)
    # FIRE's last steps rarely land on the GSD period; append the final frame
    write_trajectory(top, folder / "dpd.gsd", folder / "dpd.dcd",
                     append_positions_a=initialized_a)
    write_pymol_script(folder / "initialized.pdb", folder / "movie.pml",
                       trajectory=folder / "dpd.dcd", color_by="resn")
    write_pymol_script(folder / "minimized.pdb", folder / "minimized.pml", color_by="resn")
    return top

top = export(OUT, sim, ff, minimized_a)
print({name: int((top.resnames == name).sum()) for name in ("PSR", "PSS")}, "atoms")

{'PSR': 1142, 'PSS': 1434} atoms


## Look at it

NGLView draws every atom as licorice and keeps only the CONECT bonds (NGL
would otherwise guess bonds between atoms that overlap during DPD).
`color="chainname"` gives one color per molecule, `"resname"` one per residue
name, `"element"` the usual element colors. The movie needs
`SAVE_TRAJECTORY = True`; `save_gif(movie, n_frames, "dpd.gif")` records it
from a live notebook.

In [7]:
from phantomwalk.all_atom.visualization import save_gif, show_movie, show_structure

show_structure(OUT / "minimized.pdb", color="resname")   # PSR against PSS

In [8]:
movie = (show_movie(OUT / "initialized.pdb", OUT / "dpd.dcd", color="resname")
         if SAVE_TRAJECTORY else None)
movie

/home/joelaforet/miniconda3/envs/phantomwalk-all-atom/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
